# Balanced 250-per-genre MIDI Transformer · Colab GPU

Train a new, smaller structured model on **750 technically usable MIDI files**:
250 Classical/Baroque, 250 Romantic, and 250 Rock & Pop. Each genre has
175 training, 50 validation and 25 test files. The subset uses improved PDMX
attribution rules, composer/artist diversity, and one file per composition family.
Existing family split assignments are preserved.

This experiment keeps normalized-tempo phrase continuity, true-ending EOS labels,
16-bucket piece progress, learned relative distance bias, and release/repetition
controls. It adds training-only melodic transposition, CUDA mixed precision,
and a fixed validation-window budget covering every validation composition.

The initial model uses context **512**, width **192**, **4 layers**, **6 heads**,
batch **16** and accumulation **2**. Maximum training remains 500 epochs with
patience 50. The benchmark includes training and validation; review it and run
the smoke test before enabling full training. Actual speed depends on the GPU.

Use the **new balanced750 ZIP** and a fresh run directory. The interrupted
expanded-data run and the original CPU baseline are separate experiments.
See [the training guide](../../docs/training.md) for data selection and usage.

## 1. Select a GPU and preserve Colab's CUDA PyTorch

Choose a GPU runtime and run the check below. The installed CUDA-enabled PyTorch
must be version >=2.6,<3. Parameters remain float32. Section 5 selects BF16
autocast when supported, otherwise FP16 autocast with gradient scaling.
Colab hardware and session availability vary; see its
[official FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
import hashlib
import json
import os
import platform
import re
import shutil
import stat
import subprocess
import sys
import time
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import torch

version_match = re.match(r"^([0-9]+)[.]([0-9]+)", torch.__version__)
if version_match is None or not (2, 6) <= tuple(map(int, version_match.groups())) < (3, 0):
    raise RuntimeError(f"Keep a Colab CUDA PyTorch >=2.6,<3 runtime; found {torch.__version__}.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime before continuing.")

torch.set_default_dtype(torch.float32)
torch.set_float32_matmul_precision("highest")
gpu = torch.cuda.get_device_properties(0)
ENVIRONMENT = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_name": gpu.name,
    "gpu_vram_gib": round(gpu.total_memory / 1024**3, 2),
    "logical_cpus": os.cpu_count(),
    "parameter_dtype": str(torch.get_default_dtype()),
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "training_precision": "resolved in section 5",
}
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Mount persistent checkpoint storage

Run this explicit authorization cell to mount your Drive. Only run artifacts
go in the persistent run directory. The dataset and prepared windows are staged
on the Colab VM to avoid many small mounted-Drive reads. This follows Colab's
[file I/O examples](https://colab.research.google.com/notebooks/io.ipynb) and
[Drive guidance](https://research.google.com/colaboratory/faq.html).

The maximum epoch budget does not guarantee a single session will finish it.
`best.pt` and `last.pt` are saved after completed epochs. Reconnect and resume
if the session ends; work from an unfinished epoch may need to be repeated.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
RUN_DIR = DRIVE_ROOT / "cadence-training/balanced250_structured_v1"
CACHE_ROOT = Path("/content/cadence-balanced250-cache")
SMOKE_ROOT = Path("/content/cadence-balanced250-smoke")
print("Persistent run directory:", RUN_DIR)
print("Local preparation cache root:", CACHE_ROOT)

## 3. Select the new balanced input ZIP

Put `cadence_balanced250_colab_bundle.zip` in
`My Drive/cadence-training/` and leave it compressed. Drive input is the default.
The notebook verifies every archive payload before installing project source,
and later checks the exact 250-per-genre split contract and selection policy.

For a local browser upload, set `BUNDLE_SOURCE = "upload"`. The expanded-data ZIP
and the original baseline ZIP are incompatible with this experiment's selection.
Mount Drive explicitly in the preceding cell.

In [ ]:
BUNDLE_SOURCE = "drive"  # "upload" or "drive"
DRIVE_BUNDLE_PATH = DRIVE_ROOT / "cadence-training/cadence_balanced250_colab_bundle.zip"
EXPECTED_BUNDLE_SHA256 = ""  # Optional: paste the ZIP hash from your local builder report.
STAGE_ROOT = Path("/content/cadence-balanced250")
INPUT_DIR = Path("/content/cadence-balanced250-input")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_ARCHIVE = INPUT_DIR / "cadence_balanced250_colab_bundle.zip"

if BUNDLE_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("Upload exactly one generated .zip input bundle.")
    BUNDLE_ARCHIVE.write_bytes(next(iter(uploaded.values())))
    del uploaded
elif BUNDLE_SOURCE == "drive":
    if not DRIVE_BUNDLE_PATH.is_file():
        raise FileNotFoundError(f"Set DRIVE_BUNDLE_PATH to your input ZIP: {DRIVE_BUNDLE_PATH}")
    shutil.copyfile(DRIVE_BUNDLE_PATH, BUNDLE_ARCHIVE)
else:
    raise ValueError("BUNDLE_SOURCE must be 'upload' or 'drive'.")

def file_sha256(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()

BUNDLE_SHA256 = file_sha256(BUNDLE_ARCHIVE)
if EXPECTED_BUNDLE_SHA256:
    expected = EXPECTED_BUNDLE_SHA256.strip().lower()
    if not re.fullmatch(r"[0-9a-f]{64}", expected) or expected != BUNDLE_SHA256:
        raise ValueError("The input ZIP does not match EXPECTED_BUNDLE_SHA256.")
print("Input ZIP SHA-256:", BUNDLE_SHA256)
print("Compressed bytes:", BUNDLE_ARCHIVE.stat().st_size)

### Validate before installing uploaded source

The bootstrap below is part of this notebook, rather than Python loaded from
the ZIP. It rejects unsafe or duplicate member paths, symlinks, encrypted
entries, inconsistent inventory, and oversized archives. It verifies every
file's size and SHA-256 before extracting project code. Use your own bundle:
these checks do not make arbitrary third-party source trustworthy.
The project validator and selection audit check approval, genres, splits,
composition groups, and manifest-to-MIDI hashes after installation and before
preparation or training.

On a same-runtime rerun, an existing stage is reused only after its recorded
files are checked against the uploaded bundle again. To change bundle source,
start a fresh runtime or choose a new stage path before importing the project.

In [ ]:
def unique_json_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result

def canonical_member(name):
    relative = PurePosixPath(name)
    if (not name or chr(92) in name or ":" in name or relative.is_absolute()
            or any(part.startswith(".") for part in relative.parts)
            or any(ord(char) < 32 for char in name) or name != relative.as_posix()):
        raise ValueError(f"Unsafe ZIP member: {name!r}")
    project_source = name.startswith("project/src/musicgen/") and (
        relative.suffix == ".py"
        or name == "project/src/musicgen/generation/runtime/provenance.json"
        or ("/frontend/templates/" in name and relative.suffix == ".html")
        or ("/frontend/static/css/" in name and relative.suffix == ".css")
        or ("/frontend/static/js/" in name and relative.suffix == ".js")
    )
    if not (name in {"bundle.json", "dataset/selected.csv", "project/pyproject.toml", "project/README.md"}
            or project_source
            or (name.startswith("dataset/") and relative.suffix.lower() in {".mid", ".midi"})):
        raise ValueError(f"Unexpected ZIP member: {name!r}")
    return relative

def stage_verified_bundle(archive_path, destination):
    max_members, max_total, max_member = 50_000, 4_000_000_000, 100_000_000
    destination = Path(destination)
    if destination.is_symlink():
        raise ValueError("Stage root must not be a symlink.")
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        if len(infos) > max_members or sum(info.file_size for info in infos) > max_total:
            raise ValueError("Bundle exceeds the member or expanded-size limit.")
        names = set()
        for info in infos:
            canonical_member(info.filename)
            mode = stat.S_IFMT(info.external_attr >> 16)
            if (info.orig_filename != info.filename or info.filename in names or info.is_dir() or info.flag_bits & 1
                    or mode not in (0, stat.S_IFREG) or info.file_size > max_member):
                raise ValueError(f"Invalid, duplicate, encrypted, or oversized member: {info.filename}")
            names.add(info.filename)
        if "bundle.json" not in names or archive.getinfo("bundle.json").file_size > 16_000_000:
            raise ValueError("Missing or oversized bundle.json.")
        record_bytes = archive.read("bundle.json")
        record = json.loads(record_bytes, object_pairs_hook=unique_json_object)
        expected_layout = {
            "format": "cadence-colab-v1", "project_root": "project",
            "dataset_root": "dataset", "manifest_path": "dataset/selected.csv",
        }
        if not isinstance(record, dict) or any(record.get(k) != v for k, v in expected_layout.items()):
            raise ValueError("Unsupported bundle format or layout.")
        inventory = record.get("files")
        if not isinstance(inventory, dict) or set(inventory) != names - {"bundle.json"}:
            raise ValueError("ZIP contents do not match the checksum inventory.")
        required = {
            "project/pyproject.toml", "project/README.md", "dataset/selected.csv",
            "project/src/musicgen/__init__.py",
            "project/src/musicgen/training/colab.py",
            "project/src/musicgen/training/training.py",
        }
        if not required <= inventory.keys():
            raise ValueError("Bundle is missing required project or dataset files.")
        for name, item in inventory.items():
            if (not isinstance(item, dict) or type(item.get("size")) is not int
                    or item["size"] != archive.getinfo(name).file_size
                    or not isinstance(item.get("sha256"), str)
                    or not re.fullmatch(r"[0-9a-f]{64}", item["sha256"])):
                raise ValueError(f"Invalid inventory entry: {name}")
            checksum = hashlib.sha256()
            with archive.open(name) as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b""):
                    checksum.update(block)
            if checksum.hexdigest() != item["sha256"]:
                raise ValueError(f"Checksum mismatch: {name}")
        if inventory["dataset/selected.csv"]["sha256"] != record.get("manifest_sha256"):
            raise ValueError("Manifest checksum does not match bundle metadata.")

        if destination.exists():
            for existing in destination.rglob("*"):
                relative = existing.relative_to(destination)
                generated = (
                    "__pycache__" in relative.parts and existing.suffix == ".pyc"
                ) or any(part.endswith(".egg-info") for part in relative.parts)
                if existing.is_symlink() or (
                    existing.is_file() and relative.as_posix() not in names and not generated
                ):
                    raise ValueError("Existing stage contains unrecorded files; use a fresh stage/runtime.")
                if generated and existing.is_file() and existing.suffix == ".pyc":
                    existing.unlink()  # Rebuild bytecode from the verified source below.
            for name in names:
                target = destination.joinpath(*PurePosixPath(name).parts)
                if any(parent.is_symlink() for parent in (target, *target.parents)):
                    raise ValueError(f"Existing stage contains a symlink: {name}")
                expected_hash = (hashlib.sha256(record_bytes).hexdigest() if name == "bundle.json"
                                 else inventory[name]["sha256"])
                if not target.is_file() or file_sha256(target) != expected_hash:
                    raise ValueError("Existing stage differs from this bundle; use a fresh stage/runtime.")
        else:
            destination.mkdir(parents=True)
            try:
                for info in infos:
                    target = destination.joinpath(*PurePosixPath(info.filename).parts)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(info) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            except Exception:
                shutil.rmtree(destination)
                raise
    return record

BUNDLE_RECORD = stage_verified_bundle(BUNDLE_ARCHIVE, STAGE_ROOT)
PROJECT_ROOT = STAGE_ROOT / "project"
DATASET_ROOT = STAGE_ROOT / "dataset"
MANIFEST_PATH = DATASET_ROOT / "selected.csv"
print("Verified project:", PROJECT_ROOT)
print("Verified manifest rows:", BUNDLE_RECORD.get("audit", {}).get("rows"))

## 4. Install the verified project without replacing PyTorch

Only non-Torch dependencies are installed here. The editable install uses
`--no-deps` to preserve Colab's CUDA-enabled PyTorch. The project is then checked
again with its bundle validator and manifest audit. Source or dataset changes
require a new experiment and preparation fingerprint.

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "pydantic-settings>=2,<3", "prometheus-client>=0.21,<1",
    "packaging>=24,<27", "numpy>=1.26,<3", "matplotlib>=3.9,<4",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(PROJECT_ROOT),
], check=True)
# Make the editable source available in this already-running notebook kernel.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dataclasses import asdict

from musicgen.generation.runtime.tokenizer import GENRES, VOCAB_SHA
from musicgen.training.colab import validate_bundle
from musicgen.training.manifest import audit_manifest
from musicgen.training.training import (
    TrainingConfig,
    benchmark,
    evaluate_test,
    export_checkpoint,
    generate_samples,
    prepare_dataset,
    smoke_train,
    train,
)

checked_record = validate_bundle(BUNDLE_ARCHIVE)
print("Bundle format:", checked_record["format"])
print("Vocabulary SHA-256:", VOCAB_SHA)
print("PyTorch preserved:", torch.__version__, "CUDA:", torch.cuda.is_available())
import inspect
if not Path(inspect.getfile(TrainingConfig)).resolve().is_relative_to(PROJECT_ROOT / "src"):
    raise RuntimeError("Restart the runtime and rerun setup to load this bundle's training source.")

## 5. Configure the smaller experiment or resume its own checkpoint

The new defaults reduce context/model cost, increase the physical batch and use
CUDA automatic mixed precision. BF16 is selected on supporting GPUs; FP16 uses
a saved/restored GradScaler. Model weights remain float32 for CPU export.
Training augmentation transposes melodic pitches by up to two semitones; drums,
rhythm and validation/test data remain unchanged.

Validation uses a fixed first/last/evenly-spaced sample of at most 16 windows
per file every epoch. Every validation composition and genre remains represented.
The selected window count and index fingerprint are recorded. Final test
evaluation uses the complete test split.

Resume restores the exact saved dataset/configuration/optimizer/scaler and random
states. The new Drive run directory avoids resuming the interrupted large run.
For an out-of-memory error, try batch 8 and accumulation 4 before starting a new
run. A different context requires preparing a matching cache.

In [ ]:
NEW_RUN_CONFIG = TrainingConfig(
    context=512, d_model=192, layers=4, heads=6, dropout=0.15,
    batch_size=16, grad_accum=2, epochs=500, lr=3e-4,
    weight_decay=0.01, patience=50, min_delta=0.0001,
    seed=42, torch_threads=min(4, os.cpu_count() or 1),
    max_windows_per_file=16, device="cuda", smoke=False,
    progress_bins=16, relative_attention=True, transpose_semitones=2,
    val_windows_per_file=16,
    precision="amp_bfloat16" if torch.cuda.is_bf16_supported(including_emulation=False) else "amp_float16",
)
RESUME_EXISTING_RUN = True
RESUME_FROM = None
last_checkpoint = RUN_DIR / "last.pt"
if last_checkpoint.is_file():
    if not RESUME_EXISTING_RUN:
        raise FileExistsError("Enable resume or choose a new RUN_DIR.")
    saved = torch.load(last_checkpoint, map_location="cpu", weights_only=True)
    config = TrainingConfig(**saved["signature"]["training_config"])
    if (config.device != "cuda" or config.smoke or config.progress_bins != 16
            or not config.relative_attention or config.transpose_semitones != 2
            or config.val_windows_per_file != 16):
        raise ValueError("This directory does not contain the balanced structured experiment.")
    RESUME_FROM = last_checkpoint
    del saved
    print("Resuming the saved configuration; new-run defaults are not applied.")
else:
    config = NEW_RUN_CONFIG
    print("Configured a new balanced 250-per-genre run.")
if config.precision == "amp_bfloat16" and not torch.cuda.is_bf16_supported(including_emulation=False):
    raise ValueError("The saved run needs a native BF16 GPU. Reconnect to compatible hardware "
                     "or choose a fresh run directory for an FP16 experiment.")
ENVIRONMENT.update(training_precision=config.precision,
                   automatic_mixed_precision=config.precision != "float32")
print(json.dumps(asdict(config), indent=2))
print("Resume checkpoint:", RESUME_FROM)

## 6. Verify the exact balanced source selection and prepare windows

The manifest must contain 250 files per genre with 175/50/25 train/validation/test
files, one per composition family, under the balanced selection policy. Source
hashes and all selected MIDI are checked again. Preparation requires exactly
250 usable files per genre; a technical reject stops the run for a rebuilt
selection rather than silently leaving 249.

Tempo normalization preserves score-beat phrases across rubato changes. True
piece endings receive EOS targets; metadata/silence splits do not create false
endings. Progress features survive complete-event framing. Dataset windows stay
in SQLite on the Colab VM, while the audit and preparation summary persist in
Drive. Source attribution remains provisional.

In [ ]:
import csv

with MANIFEST_PATH.open(encoding="utf-8-sig", newline="") as handle:
    selected_rows = list(csv.DictReader(handle))
if len(selected_rows) != 750 or any(not row.get("selection_policy") for row in selected_rows):
    raise ValueError("Use the new balanced250 bundle, rather than a previous training ZIP.")
from musicgen.training.balanced import SELECTION_POLICY
from musicgen.training.pdmx import RULES_VERSION
expected_splits = {"train": 175, "val": 50, "test": 25}
if len(selected_rows) != 750 or any(
    row.get("selection_policy") != SELECTION_POLICY
    or row.get("rules_version") != RULES_VERSION
    or row.get("authorization_basis") != "user_authorized_expanded_training_selection"
    or row.get("classical_scope") != "baroque_classical"
    for row in selected_rows
):
    raise ValueError("Use the new balanced250 source bundle with 750 selected MIDI files.")
counts = Counter((row["genre"], row["split"]) for row in selected_rows)
if any(counts[genre, split] != count
       for genre in GENRES for split, count in expected_splits.items()):
    raise ValueError("Expected 175/50/25 files per genre in the balanced selection.")
if len({row["composition_group"] for row in selected_rows}) != 750:
    raise ValueError("Balanced selection must use one file per composition family.")
del selected_rows

audit = audit_manifest(
    MANIFEST_PATH, DATASET_ROOT,
    min_per_genre=250, max_per_genre=250, verify_hashes=True,
)
print(json.dumps(audit, indent=2, default=str))
CACHE_DIR = CACHE_ROOT / f"context{config.context}-progress{config.progress_bins}"
prepared = prepare_dataset(
    MANIFEST_PATH, DATASET_ROOT, CACHE_DIR,
    context=config.context, min_per_genre=250, max_per_genre=250,
    storage="sqlite", normalize_tempo=True, progress_bins=config.progress_bins,
)
if prepared["metadata"]["dataset_kind"] != "PDMX_CURATED_MIDI":
    raise ValueError("Expected the expanded PDMX selection for this experiment.")
report = prepared["report"]
summary = {key: value for key, value in report.items()
           if key not in {"file_reports", "rejected_files"}}
summary["rejected_count"] = len(report["rejected_files"])
summary["rejected_examples"] = report["rejected_files"][:10]
conversion_totals = Counter()
conversion_keys = (
    "input_notes", "retained_notes", "long_durations_clipped",
    "notes_clipped_at_metadata_change", "notes_in_unsupported_meter_segments",
    "same_position_notes_collapsed", "same_pitch_overlaps_truncated",
    "tempo_values_quantized", "long_silence_splits",
)
for file_report in report["file_reports"]:
    conversion_totals.update({key: file_report.get(key, 0) for key in conversion_keys})
summary["conversion_totals"] = dict(conversion_totals)
print(json.dumps(summary, indent=2, default=str))
print("Dataset SHA-256:", prepared["dataset_sha256"])
print("Local per-file report:", CACHE_DIR / "preparation_report.json")
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
preparation_summary = {
    "identity": prepared["identity"], "dataset_sha256": prepared["dataset_sha256"],
    "report": summary, "audit": audit,
}
(RUN_DIR.parent / f"{RUN_DIR.name}.preparation.json").write_text(
    json.dumps(preparation_summary, indent=2, default=str) + "\n"
)

## 7. Benchmark training and validation on the assigned GPU

The helper synchronizes CUDA and includes loader work. It estimates training
and the actual capped validation protocol separately. The combined epoch
estimate excludes checkpoint writes to Drive and can differ from observed
epoch timing. The default full-training gate below rejects an estimate above
five minutes per epoch; adjust the configuration or explicitly change that
budget after reviewing the measurement.

In [ ]:
cuda_estimate = benchmark(prepared, config, steps=5)
print(json.dumps(cuda_estimate, indent=2, default=str))
estimated_epoch_minutes = cuda_estimate["estimated_epoch_seconds"] / 60
print(f"Estimated training + validation per epoch: {estimated_epoch_minutes:.2f} minutes")
print(f"Estimated maximum compute for {config.epochs} epochs: "
      f"{cuda_estimate['estimated_epoch_seconds'] * config.epochs / 3600:.2f} hours")
print("Checkpoint I/O and Colab load add time. Early stopping can shorten the run.")
(RUN_DIR.parent / f"{RUN_DIR.name}.benchmark.json").write_text(
    json.dumps(cuda_estimate, indent=2, default=str) + "\n"
)
torch.cuda.empty_cache()

## 8. Smoke-test the selected CUDA precision

Run one small structured-model epoch using the configured BF16/FP16 mode, gradient
accumulation, validation and checkpoint writing. The smoke checkpoint is marked
as a pipeline test and cannot be exported as an application model. This check
must pass on the assigned GPU before starting the full run.

In [ ]:
smoke_dir = SMOKE_ROOT / f"run_{time.time_ns()}"
smoke_run = smoke_train(prepared, smoke_dir, device="cuda", precision=config.precision)
print(json.dumps({key: value for key, value in smoke_run.items() if key != "history"}, indent=2))
torch.cuda.empty_cache()

## 9. Train or resume the balanced experiment

Review the audit, exact source counts, combined epoch estimate and smoke test.
Set `RUN_FULL_TRAINING = True` when ready. The configurable estimate budget is
five minutes per epoch; it is a preflight guard, not a timing guarantee.

`best.pt` follows fixed-protocol validation selection, `last.pt` includes optimizer,
precision/scaler and random state for resume, and history records training and validation timings. Drive checkpoint writes add time outside those measurements.
Disconnecting can lose the unfinished epoch.
Keep the same source bundle and run directory when resuming this experiment.

In [ ]:
RUN_FULL_TRAINING = False
MAX_ESTIMATED_EPOCH_MINUTES = 5.0
full_run = None
if RUN_FULL_TRAINING:
    if cuda_estimate["estimated_epoch_seconds"] > MAX_ESTIMATED_EPOCH_MINUTES * 60:
        raise RuntimeError("Estimated epoch exceeds the configured budget. Review the benchmark "
                           "and adjust the configuration or budget before starting.")
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    environment_log = RUN_DIR.parent / f"{RUN_DIR.name}.sessions.jsonl"
    session = {
        **ENVIRONMENT, "bundle_sha256": BUNDLE_SHA256,
        "dataset_sha256": prepared["dataset_sha256"],
        "configuration": asdict(config),
        "resume_from": str(RESUME_FROM) if RESUME_FROM else None,
    }
    with environment_log.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(session) + chr(10))
    full_run = train(prepared, RUN_DIR, config, resume_from=RESUME_FROM)
    print(json.dumps({key: value for key, value in full_run.items() if key != "history"}, indent=2))
else:
    print("Full CUDA training is disabled. Enable it after reviewing the checks above.")

## 10. Review learning curves and validation selection

Training loss is token-weighted. Validation averages file-group losses within
each genre, then weights the three genres equally, so absolute training and
validation values have different weighting. Inspect their trends and per-genre
validation curves. Loss alone does not demonstrate coherent or genre-accurate
music. Keep test results out of configuration and checkpoint selection.

This run's validation is a fixed, composition-covering window sample. Compare
curves within this protocol; the numeric loss need not be directly comparable
to earlier runs that evaluated every validation window. Final test evaluation
uses all test windows. Training transposition is disabled during evaluation.

In [ ]:
import matplotlib.pyplot as plt

history_path = RUN_DIR / "history.json"
if full_run is not None:
    history = full_run["history"]
elif history_path.is_file():
    history = json.loads(history_path.read_text())["history"]
else:
    history = []
SELECTED_CHECKPOINT = RUN_DIR / "best.pt" if (RUN_DIR / "best.pt").is_file() else None
if history:
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="Training")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Next-token loss", title="Learning curves")
    axes[0].legend()
    for genre in GENRES:
        axes[1].plot(epochs, [row["val_per_genre"][genre] for row in history], label=genre)
    axes[1].set(xlabel="Epoch", ylabel="Validation loss", title="Validation by genre")
    axes[1].legend()
    plt.show()
    print("Validation-selected checkpoint:", SELECTED_CHECKPOINT)
else:
    print("No completed epochs are recorded for this run yet.")

## 11. Compare one-minute samples and endings

Generate 60-second samples for every genre with seeds 7 and 19. Listen for
phrase development, excessive looping, genre identity, and a convincing ending.
Use the same seed set and renderer when comparing checkpoints; do not tune
against the test split.

The renderer separately applies a gentle repeated-event penalty and a documented
release ending (lower onset density and velocity, final sonority and a short
silent tail inside the requested duration). This is an inference policy, **not
proof of a learned harmonic cadence**. Metadata records it along with descriptive
structure metrics. Progress conditioning and real-ending training must still be
judged by listening.

Sample names include an inference-source/profile fingerprint so cached samples
from an older rendering policy cannot masquerade as a fresh comparison.

In [ ]:
GENERATE_SAMPLES = False
sample_metrics = None
if GENERATE_SAMPLES:
    if SELECTED_CHECKPOINT is None:
        raise RuntimeError("Select a full-run checkpoint before generating samples.")
    sample_metrics = generate_samples(
        SELECTED_CHECKPOINT, RUN_DIR / "samples_60s", seconds=60,
        seeds=(7, 19), device="cuda",
    )
    print(json.dumps(sample_metrics, indent=2, default=str))
    print("Sample directory:", RUN_DIR / "samples_60s")
else:
    print("GPU sample generation is disabled until a checkpoint is ready for review.")

## 12. Evaluate the held-out test split once

Freeze model-selection choices first, then enable this cell. Results are saved
in `test_metrics.json`, reused for the same checkpoint and dataset, and rejected
if that run directory already records another checkpoint's test result. Do not
tune epochs, architecture, data selection, or labels against this score.

In [ ]:
RUN_TEST_EVALUATION = False
test_metrics = None
if RUN_TEST_EVALUATION:
    if SELECTED_CHECKPOINT is None:
        raise RuntimeError("Select a validation checkpoint before the final test evaluation.")
    test_metrics = evaluate_test(SELECTED_CHECKPOINT, prepared, device="cuda")
    print(json.dumps(test_metrics, indent=2, default=str))
else:
    print("Held-out evaluation is disabled until model selection is final.")

## 13. Export a compatible inference checkpoint

Enable export after reviewing final evaluation and listening observations.
The helper validates the checkpoint with the actual runtime and exports the
required inference fields in a CPU-loadable file. It rejects smoke checkpoints
and refuses to overwrite an existing destination. Choose a new versioned path
if you intentionally export a different selected model.

Keep `last.pt` for training resume and export `best.pt` for inference. The
exported file is private; place it at `models/best.pt` on your inference host.
State dictionaries and `weights_only=True` are described in the
[PyTorch serialization notes](https://docs.pytorch.org/docs/2.6/notes/serialization.html).

In [ ]:
EXPORT_CHECKPOINT = False
EXPORTED_CHECKPOINT = RUN_DIR / "export/best.pt"
if EXPORT_CHECKPOINT:
    if SELECTED_CHECKPOINT is None or test_metrics is None:
        raise RuntimeError("Select a full checkpoint and review its final test evaluation first.")
    exported = export_checkpoint(SELECTED_CHECKPOINT, EXPORTED_CHECKPOINT)
    print("Exported inference checkpoint:", exported)
else:
    print("Export is disabled; no inference checkpoint is copied.")

### Download when ready

Run this separate opt-in cell to download the exported checkpoint through your
browser, or retrieve it from the persistent Drive folder later. Downloads and
Drive access follow Colab's [official I/O examples](https://colab.research.google.com/notebooks/io.ipynb).

In [ ]:
DOWNLOAD_CHECKPOINT = False
if DOWNLOAD_CHECKPOINT:
    from google.colab import files

    if not EXPORTED_CHECKPOINT.is_file():
        raise FileNotFoundError("Export the selected inference checkpoint first.")
    files.download(str(EXPORTED_CHECKPOINT))
else:
    print("Browser download is disabled; Drive retains saved run artifacts.")

## Keep evidence with the experiment

Record the selected-manifest and rule versions, exact source/split counts,
composer diversity, preparation conversions, GPU/PyTorch environment, resolved
precision, actual timings, validation protocol, curves and listening notes.
Technical success and balanced file counts do not establish an absence of bias
or a musical-quality improvement.

The earlier training sampler already equalized sampled genre windows. This
experiment also equalizes selected file/split counts and reduces compute so
we can iterate on genre identity, repetition and endings. The release policy
still supplies an explicit ending envelope rather than a learned tonic cadence.

See [PyTorch AMP examples](https://docs.pytorch.org/docs/2.6/notes/amp_examples.html)
for autocast, scaling and accumulated-gradient handling;
[Music Transformer](https://arxiv.org/abs/1809.04281) for relative-attention context;
and [PDMX documentation](https://github.com/pnlong/PDMX) for source metadata.
The relative distance bias here is a smaller additive learned table.

Datasets, input bundles, model weights, generated MIDI and logs stay out of Git.
The notebook ships with cleared outputs. Evaluate and listen before replacing
an application checkpoint.